# Sintesi Automatica di Cartelle Cliniche

Questo notebook implementa un sistema automatizzato per generare sintesi leggibili delle cartelle cliniche di pazienti ospedalieri, utilizzando modelli pre-addestrati di Hugging Face.

**Flusso:**
1. Download del file JSON con le cartelle cliniche
2. Costruzione del testo di input per ogni paziente
3. Generazione della sintesi con `facebook/bart-large-cnn`
4. Salvataggio dell'output in JSON

## 1. Installazione delle dipendenze

In [ ]:
!pip install -q transformers torch sentencepiece

## 2. Import delle librerie

In [ ]:
import json
import urllib.request
import torch
from transformers import pipeline

device = 0 if torch.cuda.is_available() else -1
print(f"Dispositivo in uso: {'GPU' if device == 0 else 'CPU'}")

## 3. Download del dataset

In [ ]:
DATA_URL = (
    "https://raw.githubusercontent.com/Profession-AI/progetti-deeplearning/"
    "refs/heads/main/Sintesi%20automatica%20di%20cartelle%20cliniche%20di%20un"
    "%E2%80%99azienda%20ospedaliera/hospital_records.json"
)
INPUT_FILE  = "hospital_records.json"
OUTPUT_FILE = "sintesi_pazienti.json"

urllib.request.urlretrieve(DATA_URL, INPUT_FILE)
print(f"File scaricato: {INPUT_FILE}")

with open(INPUT_FILE, "r", encoding="utf-8") as f:
    hospital_records = json.load(f)

print(f"Pazienti caricati: {len(hospital_records)}")
# Anteprima della struttura
first_patient = hospital_records[0]
print("\nCampi disponibili per paziente:", list(first_patient.keys()))
if "ricoveri" in first_patient:
    print("Campi per ricovero:", list(first_patient["ricoveri"][0].keys()))

## 4. Costruzione del testo di input per ogni paziente

Concateniamo le informazioni cliniche rilevanti di tutti i ricoveri in un unico testo strutturato che verrà poi riassunto dal modello.

In [ ]:
def build_patient_text(patient: dict) -> str:
    """
    Costruisce una stringa con le informazioni cliniche rilevanti
    di tutti i ricoveri di un paziente.
    """
    lines = []

    # Dati anagrafici
    nome = patient.get("nome", patient.get("name", "N/D"))
    eta  = patient.get("eta",  patient.get("age",  "N/D"))
    lines.append(f"Patient: {nome}, age {eta}.")

    ricoveri = patient.get("ricoveri", patient.get("admissions", []))

    for i, ricovero in enumerate(ricoveri, start=1):
        lines.append(f"Admission {i}:")

        # Diagnosi
        diagnosi = ricovero.get("diagnosi", ricovero.get("diagnosis", ""))
        if diagnosi:
            lines.append(f"  Diagnosis: {diagnosi}.")

        # Anamnesi
        anamnesi = ricovero.get("anamnesi", ricovero.get("anamnesis", ""))
        if anamnesi:
            lines.append(f"  Medical history: {anamnesi}.")

        # Prognosi
        prognosi = ricovero.get("prognosi", ricovero.get("prognosis", ""))
        if prognosi:
            lines.append(f"  Prognosis: {prognosi}.")

        # Date
        data_in  = ricovero.get("data_ricovero",  ricovero.get("admission_date",  ""))
        data_out = ricovero.get("data_dimissione", ricovero.get("discharge_date", ""))
        if data_in or data_out:
            lines.append(f"  Period: from {data_in} to {data_out}.")

        # Medicinali
        medicinali = ricovero.get("medicinali", ricovero.get("medications", []))
        if medicinali:
            if isinstance(medicinali, list):
                medicinali = ", ".join(medicinali)
            lines.append(f"  Medications: {medicinali}.")

        # Referti
        referti = ricovero.get("referti", ricovero.get("reports", ""))
        if referti:
            if isinstance(referti, list):
                referti = " ".join(referti)
            lines.append(f"  Reports: {referti}.")

    return " ".join(lines)


# Test visivo sulla prima cartella
sample_text = build_patient_text(hospital_records[0])
print("--- Testo di input (paziente 1) ---")
print(sample_text)

## 5. Caricamento del modello di sintesi

Utilizziamo `facebook/bart-large-cnn`, un modello BART fine-tuned su CNN/DailyMail ottimizzato per la generazione di riassunti in lingua inglese.

In [ ]:
MODEL_NAME = "facebook/bart-large-cnn"

summarizer = pipeline(
    "summarization",
    model=MODEL_NAME,
    device=device,
)
print(f"Modello '{MODEL_NAME}' caricato.")

## 6. Generazione delle sintesi

Per ogni paziente:
- Costruiamo il testo di input
- Tronchiamo l'input a 1024 token (limite del modello BART)
- Generiamo il riassunto

In [ ]:
# Lunghezza massima input (in caratteri) prima della tokenizzazione.
# BART accetta al massimo 1024 token; ~4 caratteri/token è una stima conservativa.
MAX_INPUT_CHARS = 3000
MAX_SUMMARY_LEN = 200
MIN_SUMMARY_LEN = 60

results = []

for idx, patient in enumerate(hospital_records):
    nome = patient.get("nome", patient.get("name", f"Paziente {idx+1}"))
    patient_id = patient.get("id", patient.get("patient_id", str(idx + 1)))

    # Costruzione e troncamento del testo
    raw_text = build_patient_text(patient)
    input_text = raw_text[:MAX_INPUT_CHARS]

    print(f"[{idx+1}/{len(hospital_records)}] Elaborazione: {nome} ...", end=" ", flush=True)

    output = summarizer(
        input_text,
        max_length=MAX_SUMMARY_LEN,
        min_length=MIN_SUMMARY_LEN,
        do_sample=False,
        truncation=True,
    )
    summary = output[0]["summary_text"]
    print("OK")

    results.append({
        "patient_id": patient_id,
        "nome": nome,
        "sintesi": summary,
    })

print("\nElaborazione completata.")

## 7. Salvataggio dell'output

In [ ]:
with open(OUTPUT_FILE, "w", encoding="utf-8") as f:
    json.dump(results, f, ensure_ascii=False, indent=2)

print(f"Output salvato in: {OUTPUT_FILE}")
print(f"Pazienti elaborati: {len(results)}")

## 8. Anteprima delle sintesi generate

In [ ]:
for record in results:
    print(f"{'='*60}")
    print(f"Paziente: {record['nome']}  (ID: {record['patient_id']})")
    print(f"Sintesi: {record['sintesi']}")
print("="*60)

## 9. Download del file di output (solo in Colab)

In [ ]:
try:
    from google.colab import files
    files.download(OUTPUT_FILE)
except ImportError:
    print(f"Ambiente non Colab: scarica manualmente il file '{OUTPUT_FILE}'.")